In [ ]:
# Full notebook cell: DRKG pipeline (GCN, GAT, GraphSAGE, GIN + DistMult baseline)
# - neighbor sampling (NeighborLoader) with ClusterLoader fallback
# - inductive disease-level evaluation (reads train/val/test csvs)
# - mechanistic path regularization (Compound->Gene->Disease)
# - ensembles + MC Dropout
# - ranked CSV with supporting genes (C->G->D)
# ------------------------------------------------------------------------------
import os, json, random, math, time
from collections import defaultdict, Counter
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

# PyG imports
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader, ClusterData, ClusterLoader
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv



In [2]:
# -------------------- CONFIG --------------------
GRAPH_DIR = r"/home/manasa/Drug_Repurposing_Gnn/data/processed_graph"
OUT_DIR = os.path.join(GRAPH_DIR, "pipeline_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
set_seed = lambda s=SEED: (random.seed(s), np.random.seed(s), torch.manual_seed(s), torch.cuda.manual_seed_all(s) if torch.cuda.is_available() else None)
set_seed(SEED)

# Hyperparameters (tune)
HDIM = 128
OUTDIM = 128
EPOCHS = 6          # increase for real runs
BATCH_SIZE = 2048
NEIGHBOR_SAMPLES = [20, 10]   # per-layer neighbor sizes
LR = 1e-3
WEIGHT_DECAY = 1e-6
PATH_REG_WEIGHT = 0.5
NEG_RATIO = 1
ENSEMBLE_SIZE = 2    # number of model retrains per model family
MC_RUNS = 30         # MC dropout forward passes for uncertainty
TOP_K = 50



In [3]:
# -------------------- helper functions --------------------
def safe_read_lines(path):
    if not os.path.exists(path):
        return []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return [ln.rstrip("\n") for ln in f]

def load_edge_files(graph_dir):
    ei = os.path.join(graph_dir, "edge_index.pt")
    et = os.path.join(graph_dir, "edge_type.pt")
    assert os.path.exists(ei) and os.path.exists(et), f"edge_index.pt or edge_type.pt not found in {graph_dir}"
    edge_index = torch.load(ei)
    edge_type = torch.load(et)
    return edge_index, edge_type

def read_inductive_csv(path):
    if not os.path.exists(path):
        return []
    df = pd.read_csv(path, header=None, dtype=str)
    # handle header row if it is 'head,relation,tail'
    if df.shape[0] > 0 and [c.lower() for c in df.iloc[0].astype(str).tolist()] == ['head','relation','tail']:
        df = df.iloc[1:].reset_index(drop=True)
    triples = [(str(r[0]).strip(), str(r[1]).strip(), str(r[2]).strip()) for r in df.values if len(r)>=3]
    return triples


In [4]:

# -------------------- load files --------------------
entities_lines = safe_read_lines(os.path.join(GRAPH_DIR, "/home/manasa/Drug_Repurposing_Gnn/data/processed_graph/entities.txt"))
relation_lines = safe_read_lines(os.path.join(GRAPH_DIR, "/home/manasa/Drug_Repurposing_Gnn/data/processed_graph/relations.txt"))
edge_index, edge_type = load_edge_files(GRAPH_DIR)

train_triples = read_inductive_csv(os.path.join(GRAPH_DIR, "/home/manasa/Drug_Repurposing_Gnn/data/train_inductive.csv"))
val_triples   = read_inductive_csv(os.path.join(GRAPH_DIR, "/home/manasa/Drug_Repurposing_Gnn/data/val_inductive.csv"))
test_triples  = read_inductive_csv(os.path.join(GRAPH_DIR, "/home/manasa/Drug_Repurposing_Gnn/data/test_inductive.csv"))

print("Loaded entities:", len(entities_lines), "relations:", len(relation_lines))
print("Edge index shape:", tuple(edge_index.shape), "edge_type len:", len(edge_type))
print("Inductive triples train/val/test:", len(train_triples), len(val_triples), len(test_triples))


Loaded entities: 94046 relations: 107
Edge index shape: (2, 5261827) edge_type len: 5261827
Inductive triples train/val/test: 5261827 584648 27786


In [5]:

# -------------------- parse entities.txt (format: "gid\tLabel") --------------------
# we observed lines like: "44057\tGene::1" etc.
ent2name = {}
ent2type = {}
for ln in entities_lines:
    # expect "<gid>\\t<Label>" ; robust parse:
    parts = ln.split("\t")
    if len(parts) >= 2:
        gid_str = parts[0].strip()
        label = parts[1].strip()
        ent2name[gid_str] = label
        lab_low = label.lower()
        if "compound::" in lab_low or "drug" in lab_low or "atc::" in lab_low:
            typ = "compound"
        elif "gene::" in lab_low:
            typ = "gene"
        elif "disease::" in lab_low or "doid:" in lab_low:
            typ = "disease"
        else:
            typ = "other"
        ent2type[gid_str] = typ
    else:
        # fallback: treat the line as label only, use index
        idx = str(len(ent2name))
        ent2name[idx] = ln.strip()
        ent2type[idx] = "other"

# Build mapping from label-string (as used in CSV) to gid-str
# In your data, triples use labels like 'Compound::DB09303' or 'Gene::5352', while entities.txt lines have numeric gid keys mapping to labels.
label_to_gid = {}
for gid_str, label in ent2name.items():
    label_to_gid[label] = gid_str

# Build integer gid mapping for embeddings (int indices used by the preprocessed edge_index)
# Entities lines had integer keys (strings) corresponding to integer gid; convert to int
gidstr_to_int = {}
for gid_str in ent2name.keys():
    try:
        gid_int = int(gid_str)
    except:
        continue
    gidstr_to_int[gid_str] = gid_int

num_nodes = int(edge_index.max().item()) + 1
print("num_nodes inferred from edge_index:", num_nodes)


num_nodes inferred from edge_index: 94046


In [6]:

# -------------------- parse relations.txt (format: "rid\tRelName") --------------------
rid_to_relname = {}
for ln in relation_lines:
    parts = ln.split("\t")
    if len(parts) >= 2:
        rid = parts[0].strip()
        rname = parts[1].strip()
        rid_to_relname[rid] = rname
    else:
        # fallback: numeric line index mapping
        pass

# We also build token-based relation detection for C->G and G->D
def relname_indicates(relname, pattern_list):
    if relname is None: return False
    rl = relname.lower()
    return any(p in rl for p in pattern_list)

# patterns
pattern_cg = ["compound:gene", "compound:gene", "target", "bind", "hetionet::cbg", "dgidb::", "gnbr::a", "gnbr::b", "gnbr::e"]
pattern_gd = ["gene:disease", "gene:disease", "assoc", "associated", "gnbr::d", "gnbr::g", "gnbr::md", "gnbr::mp", "gnbr::j"]


In [7]:

# -------------------- helper: map triple (label-based) to integer gids --------------------
def map_triple_label_to_int(triple):
    h, r, t = triple
    # try matches: if h/t are like 'Compound::DB09303' directly matching entities labels
    if h in label_to_gid and t in label_to_gid:
        h_gid_str = label_to_gid[h]; t_gid_str = label_to_gid[t]
        # convert to int id if possible
        try:
            return int(h_gid_str), r, int(t_gid_str)
        except:
            return None
    # else, h/t might already be string ints or like 'Gene::5352' where entity label stored by line contains index prefixed number.
    # Try to find any entity label that endswith h or equals h
    if h in gidstr_to_int and t in gidstr_to_int:
        return gidstr_to_int[h], r, gidstr_to_int[t]
    # try numeric conversion (CSV might store integer gids already)
    try:
        hi = int(h); ti = int(t)
        if 0 <= hi < num_nodes and 0 <= ti < num_nodes:
            return hi, r, ti
    except:
        pass
    # fallback: try matching by suffix
    matched_h = None; matched_t = None
    for label, gidstr in label_to_gid.items():
        if label.endswith(h):
            try:
                matched_h = int(gidstr); break
            except: pass
    for label, gidstr in label_to_gid.items():
        if label.endswith(t):
            try:
                matched_t = int(gidstr); break
            except: pass
    if matched_h is not None and matched_t is not None:
        return matched_h, r, matched_t
    return None


In [8]:

# -------------------- build train/val/test integer triples --------------------
train_int = [map_triple_label_to_int(t) for t in train_triples]
train_int = [t for t in train_int if t is not None]
val_int   = [map_triple_label_to_int(t) for t in val_triples]; val_int = [t for t in val_int if t is not None]
test_int  = [map_triple_label_to_int(t) for t in test_triples]; test_int = [t for t in test_int if t is not None]

print("Mapped triples to integer gids:", len(train_int), len(val_int), len(test_int))


Mapped triples to integer gids: 5261827 582709 4855


In [9]:

# -------------------- extract mechanistic pairs from train triples --------------------
comp_gene_pairs = set()
gene_disease_pairs = set()
comp_disease_pairs = set()

for h, r, t in train_int:
    # map r to name if relation id string present in rid_to_relname
    rname = None
    # 'r' might already be a relation name or id; try both
    if str(r) in rid_to_relname:
        rname = rid_to_relname[str(r)]
    else:
        rname = str(r)
    # entity types via gid->label mapping
    # find gid_str for h, t
    h_gid_str = str(h); t_gid_str = str(t)
    # find type by looking up ent2name mapping: we need mapping from int->label: gidstr_to_int inverse
    # easier: build int->type map from entities_lines earlier (we have ent2type with keys as gid_str)
    h_type = ent2type.get(h_gid_str, "other")
    t_type = ent2type.get(t_gid_str, "other")
    rl = rname.lower() if rname else ""
    # direct type-based
    if h_type == "compound" and t_type == "gene":
        comp_gene_pairs.add((h, t))
    if h_type == "gene" and t_type == "disease":
        gene_disease_pairs.add((h, t))
    if h_type == "compound" and t_type == "disease":
        comp_disease_pairs.add((h, t))
    # relation-name heuristics
    if relname_indicates(rname, pattern_cg):
        # if one side is compound and the other gene, add accordingly
        if h_type=="compound" and t_type=="gene":
            comp_gene_pairs.add((h,t))
        elif t_type=="compound" and h_type=="gene":
            comp_gene_pairs.add((t,h))
    if relname_indicates(rname, pattern_gd):
        if h_type=="gene" and t_type=="disease":
            gene_disease_pairs.add((h,t))
        elif t_type=="gene" and h_type=="disease":
            gene_disease_pairs.add((t,h))
    if "treat" in rl or "indicat" in rl:
        if h_type=="compound" and t_type=="disease":
            comp_disease_pairs.add((h,t))

print("Mechanistic pairs found (train) comp->gene:", len(comp_gene_pairs), "gene->disease:", len(gene_disease_pairs), "comp->disease:", len(comp_disease_pairs))

# If counts are zero, it's likely mapping mismatches; we printed earlier diagnostics — skip training until non-zero.
if len(comp_gene_pairs)==0 or len(gene_disease_pairs)==0:
    print("Warning: mechanism pairs count is zero for comp->gene or gene->disease. Check mapping. Continuing but path reg will be inactive.")


Mechanistic pairs found (train) comp->gene: 145065 gene->disease: 74020 comp->disease: 60568


In [10]:

# -------------------- prepare labeled compound-disease pairs (pos + neg) --------------------
def build_cd_pairs_from_triples(int_triples):
    pos = []
    for h, r, t in int_triples:
        # treat a triple as compound->disease positive if types match or relation contains 'treat' etc.
        h_type = ent2type.get(str(h), "other"); t_type = ent2type.get(str(t), "other")
        if (h_type=="compound" and t_type=="disease") or ("treat" in str(r).lower()) or ("indicat" in str(r).lower()):
            pos.append((h,t))
    return list(set(pos))

train_pos_cd = build_cd_pairs_from_triples(train_int)
val_pos_cd = build_cd_pairs_from_triples(val_int)
test_pos_cd = build_cd_pairs_from_triples(test_int)
print("Pos CD pairs train/val/test:", len(train_pos_cd), len(val_pos_cd), len(test_pos_cd))

all_disease_gids = [int(k) for k,v in ent2type.items() if v=="disease"]
all_compound_gids = [int(k) for k,v in ent2type.items() if v=="compound"]

def negative_sample_for_list(comp_list, neg_ratio=1):
    negs=[]
    for c in comp_list:
        for _ in range(neg_ratio):
            if len(all_disease_gids)==0: break
            negs.append((c, random.choice(all_disease_gids)))
    return negs

train_negs = negative_sample_for_list([c for c,_ in train_pos_cd], NEG_RATIO)
train_pairs = [(c,d,1) for c,d in train_pos_cd] + [(c,d,0) for c,d in train_negs]
random.shuffle(train_pairs)

val_pairs = [(c,d,1) for c,d in val_pos_cd] + [(c,d,0) for c,d in negative_sample_for_list([c for c,_ in val_pos_cd], 1)]
test_pairs = [(c,d,1) for c,d in test_pos_cd] + [(c,d,0) for c,d in negative_sample_for_list([c for c,_ in test_pos_cd], 1)]

print("Train pairs total:", len(train_pairs), "Val pairs:", len(val_pairs), "Test pairs:", len(test_pairs))


Pos CD pairs train/val/test: 60568 6545 1668
Train pairs total: 121136 Val pairs: 13090 Test pairs: 3336


In [11]:

# -------------------- make PyG Data object (full graph) --------------------
data = Data()
data.num_nodes = num_nodes
data.edge_index = edge_index.long()
data.edge_type = edge_type.long()


In [12]:

# -------------------- neighbor loader with fallback to ClusterLoader --------------------
use_cluster = False
try:
    # we'll create a loader per-batch using input_nodes seeds (custom per-batch). For efficiency, create a global NeighborLoader for entire node set if desired.
    neighbor_template = NeighborLoader(data, num_neighbors=NEIGHBOR_SAMPLES, input_nodes=None, batch_size=4096)
    print("NeighborLoader available")
except Exception as e:
    print("NeighborLoader creation failed; switching to ClusterLoader:", e)
    use_cluster = True


NeighborLoader available


In [13]:

# -------------------- Models definitions --------------------
class BaseGNN(nn.Module):
    def __init__(self, num_nodes, hidden_dim=HDIM, outdim=OUTDIM, dropout=0.3):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, hidden_dim)
        self.dropout = dropout
    def embed(self):
        return self.node_emb.weight

class GCNModel(BaseGNN):
    def __init__(self, num_nodes, hidden_dim=HDIM, outdim=OUTDIM, dropout=0.3):
        super().__init__(num_nodes, hidden_dim, outdim, dropout)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, outdim)
    def forward(self, edge_index, edge_type=None):
        x = self.node_emb.weight
        x = self.conv1(x, edge_index)
        x = F.relu(x); x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GATModel(BaseGNN):
    def __init__(self, num_nodes, hidden_dim=HDIM, outdim=OUTDIM, heads=2, dropout=0.3):
        super().__init__(num_nodes, hidden_dim, outdim, dropout)
        self.conv1 = GATConv(hidden_dim, hidden_dim//heads, heads=heads)
        self.conv2 = GATConv(hidden_dim, outdim, heads=1)
    def forward(self, edge_index, edge_type=None):
        x = self.node_emb.weight
        x = self.conv1(x, edge_index)
        x = F.elu(x); x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class SAGEModel(BaseGNN):
    def __init__(self, num_nodes, hidden_dim=HDIM, outdim=OUTDIM, dropout=0.3):
        super().__init__(num_nodes, hidden_dim, outdim, dropout)
        self.conv1 = SAGEConv(hidden_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, outdim)
    def forward(self, edge_index, edge_type=None):
        x = self.node_emb.weight
        x = self.conv1(x, edge_index)
        x = F.relu(x); x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GINModel(BaseGNN):
    def __init__(self, num_nodes, hidden_dim=HDIM, outdim=OUTDIM, dropout=0.3):
        super().__init__(num_nodes, hidden_dim, outdim, dropout)
        nn1 = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
        self.conv1 = GINConv(nn1)
        nn2 = nn.Sequential(nn.Linear(hidden_dim, outdim), nn.ReLU(), nn.Linear(outdim, outdim))
        self.conv2 = GINConv(nn2)
    def forward(self, edge_index, edge_type=None):
        x = self.node_emb.weight
        x = self.conv1(x, edge_index)
        x = F.relu(x); x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class DistMultBaseline(nn.Module):
    def __init__(self, num_nodes, dim=OUTDIM):
        super().__init__()
        self.E = nn.Embedding(num_nodes, dim)
    def forward(self, *args, **kwargs):
        return self.E.weight


In [14]:

# -------------------- path regularization helper --------------------
def mechanistic_path_loss(embs_tensor, comp_gene, gene_disease, device, max_samples=1024):
    if len(comp_gene)==0 or len(gene_disease)==0:
        return torch.tensor(0., device=device, requires_grad=True)
    cg = random.sample(list(comp_gene), min(len(comp_gene), max_samples))
    gd = random.sample(list(gene_disease), min(len(gene_disease), max_samples))
    comp_to_genes = defaultdict(list)
    gene_to_diseases = defaultdict(list)
    for c,g in cg: comp_to_genes[c].append(g)
    for g,d in gd: gene_to_diseases[g].append(d)
    triplets=[]
    for c, genes in comp_to_genes.items():
        for g in genes:
            for d in gene_to_diseases.get(g, []):
                triplets.append((c,g,d))
                if len(triplets)>=max_samples: break
            if len(triplets)>=max_samples: break
        if len(triplets)>=max_samples: break
    if len(triplets)==0:
        return torch.tensor(0., device=device, requires_grad=True)
    c_idx = torch.tensor([t[0] for t in triplets], dtype=torch.long, device=device)
    g_idx = torch.tensor([t[1] for t in triplets], dtype=torch.long, device=device)
    d_idx = torch.tensor([t[2] for t in triplets], dtype=torch.long, device=device)
    c_v = embs_tensor[c_idx]; g_v = embs_tensor[g_idx]; d_v = embs_tensor[d_idx]
    pred_cd = torch.sum(F.normalize(c_v,dim=1)*F.normalize(d_v,dim=1), dim=1)
    cg_sim = torch.sum(F.normalize(c_v,dim=1)*F.normalize(g_v,dim=1), dim=1)
    gd_sim = torch.sum(F.normalize(g_v,dim=1)*F.normalize(d_v,dim=1), dim=1)
    mech = torch.min(cg_sim, gd_sim)
    margin = 0.1
    loss = F.relu(margin + mech - pred_cd).mean()
    return loss


In [18]:
# -------------------- CPU-safe train routine --------------------
def train_model(model, train_pairs, data, comp_gene, gene_disease, device,
                epochs=EPOCHS, batch_size=BATCH_SIZE):
    """
    Train a GNN model on train_pairs using full-graph embeddings.
    CPU-safe: does NOT require torch-sparse or pyg-lib.
    """
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    n = len(train_pairs)
    print(f"Training {model.__class__.__name__} on {n} pairs")

    for epoch in range(epochs):
        random.shuffle(train_pairs)
        model.train()
        total_loss = 0.0

        for i in range(0, n, batch_size):
            batch = train_pairs[i:i+batch_size]

            # ----------------- CPU-safe seed nodes -----------------
            seed_nodes = torch.unique(
                torch.tensor([h for h, _, _ in batch] + [t for _, _, t in batch])
            ).to(device)

            # ----------------- forward full graph -----------------
            # model computes embeddings for all nodes
            full_embs = model(
                data.edge_index.to(device),
                data.edge_type.to(device) if hasattr(data, 'edge_type') else None
            )

            # ----------------- gather batch embeddings -----------------
            gid2local = {int(n.item()): idx for idx, n in enumerate(torch.arange(data.num_nodes))}
            comps_local, dis_local, labels = [], [], []
            for c, d, lbl in batch:
                if c in gid2local and d in gid2local:
                    comps_local.append(gid2local[c])
                    dis_local.append(gid2local[d])
                    labels.append(lbl)

            if len(comps_local) == 0:
                continue

            comps_t = torch.tensor(comps_local, dtype=torch.long, device=device)
            dis_t = torch.tensor(dis_local, dtype=torch.long, device=device)
            labels_t = torch.tensor(labels, dtype=torch.float32, device=device)

            # ----------------- compute logits -----------------
            logits = torch.sum(
                F.normalize(full_embs[comps_t], dim=1) *
                F.normalize(full_embs[dis_t], dim=1),
                dim=1
            )

            # ----------------- compute BCE loss -----------------
            loss = F.binary_cross_entropy_with_logits(logits, labels_t)

            # ----------------- optional path regularization -----------------
            if PATH_REG_WEIGHT > 0 and len(comp_gene) > 0 and len(gene_disease) > 0:
                pr = mechanistic_path_loss(full_embs, comp_gene, gene_disease, device)
                loss = loss + PATH_REG_WEIGHT * pr

            # ----------------- optimizer step -----------------
            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item() * len(comps_local)

        avg_loss = total_loss / max(1, n)
        print(f"Epoch {epoch+1}/{epochs} avg_loss={avg_loss:.6f}")

    return model


In [19]:

# -------------------- MC dropout prediction for a single model --------------------
def mc_predict(model, data, pairs, device, mc_runs=MC_RUNS):
    model.to(device)
    model.train()  # enable dropout
    preds_list = []
    with torch.no_grad():
        for _ in range(mc_runs):
            embs = model(data.edge_index.to(device), data.edge_type.to(device) if hasattr(data,'edge_type') else None).cpu().numpy()
            preds = []
            for c,d,l in pairs:
                sc = float(np.dot(embs[c] / (np.linalg.norm(embs[c])+1e-12), embs[d] / (np.linalg.norm(embs[d])+1e-12)))
                preds.append(sc)
            preds_list.append(preds)
    arr = np.stack(preds_list, axis=0)  # [mc_runs, num_pairs]
    mean = arr.mean(axis=0); var = arr.var(axis=0)
    labels = np.array([l for _,_,l in pairs])
    return mean, var, labels


In [20]:

# -------------------- ensemble wrapper --------------------
def train_and_ensemble(model_ctor, name, ensemble_size=ENSEMBLE_SIZE):
    members = []
    for e in range(ensemble_size):
        seed = SEED + e*17 + (0 if name=="gcn" else e)
        set_seed(seed)
        model = model_ctor()
        print(f"\nTraining {name} member {e+1}/{ensemble_size}")
        model = train_model(model, train_pairs, data, comp_gene_pairs, gene_disease_pairs,
                    DEVICE, epochs=EPOCHS, batch_size=BATCH_SIZE)
        members.append(model)
        torch.save(model.state_dict(), os.path.join(OUT_DIR, f"{name}_member{e}.pt"))
    return members


In [ ]:

# -------------------- train ensembles for each model family --------------------
model_families = {
    "gcn": lambda: GCNModel(num_nodes, HDIM, OUTDIM, dropout=0.3),
    "gat": lambda: GATModel(num_nodes, HDIM, OUTDIM, heads=2, dropout=0.3),
    "sage": lambda: SAGEModel(num_nodes, HDIM, OUTDIM, dropout=0.3),
    "gin": lambda: GINModel(num_nodes, HDIM, OUTDIM, dropout=0.3),
    "distmult": lambda: DistMultBaseline(num_nodes, dim=OUTDIM)
}

trained_ensembles = {}
results = {}

for name, ctor in model_families.items():
    print("\n==============================")
    print("Working on family:", name)
    members = train_and_ensemble(ctor, name, ensemble_size=ENSEMBLE_SIZE)
    trained_ensembles[name] = members
    # evaluate ensemble on test_pairs: average member MC means
    all_means = []
    all_vars = []
    for m in members:
        mean, var, labels = mc_predict(m, data, test_pairs, DEVICE, mc_runs=MC_RUNS)
        all_means.append(mean); all_vars.append(var)
    all_means = np.stack(all_means, axis=0)
    ensemble_mean = all_means.mean(axis=0)
    try:
        auroc = roc_auc_score(labels, ensemble_mean)
        auprc = average_precision_score(labels, ensemble_mean)
    except Exception as e:
        auroc = float("nan"); auprc = float("nan")
    results[name] = {"auroc": float(auroc), "auprc": float(auprc)}
    print(f"Ensemble {name} AUROC={auroc:.4f} AUPRC={auprc:.4f}")

# save results
with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
print("Saved results.json in", OUT_DIR)




Working on family: gcn

Training gcn member 1/2
Training GCNModel on 121136 pairs


In [ ]:
# -------------------- produce ranked CSV per held-out disease with supporting genes --------------------
# Build support maps from training mechanistic pairs (use integer gids)
compound_to_genes = defaultdict(set)
for c,g in comp_gene_pairs:
    compound_to_genes[c].add(g)
gene_to_diseases = defaultdict(set)
for g,d in gene_disease_pairs:
    gene_to_diseases[g].add(d)

# pick ensemble embeddings by averaging member embeddings across all trained models (all families)
# gather embeddings from all model members
all_member_embs = []
for fam, members in trained_ensembles.items():
    for m in members:
        emb = m(data.edge_index.to(DEVICE), data.edge_type.to(DEVICE) if hasattr(data,'edge_type') else None).cpu().numpy()
        all_member_embs.append(emb)
if len(all_member_embs)==0:
    print("No embeddings found; skipping ranking.")
else:
    avg_emb = np.mean(np.stack(all_member_embs, axis=0), axis=0)  # [N, D]
    # held-out diseases: derive from test_pos_cd
    held_out_diseases = sorted(list(set([d for c,d in test_pos_cd])))
    if len(held_out_diseases)==0:
        held_out_diseases = sorted(list(set([d for c,d,l in test_pairs if l==1])))
    rows=[]
    for d in tqdm(held_out_diseases):
        vec_d = avg_emb[d]
        candidate_comps = all_compound_gids
        scores = []
        for c in candidate_comps:
            sc = float(np.dot(avg_emb[c] / (np.linalg.norm(avg_emb[c])+1e-12), vec_d / (np.linalg.norm(vec_d)+1e-12)))
            scores.append((c, sc))
        scores.sort(key=lambda x: x[1], reverse=True)
        for rank, (c, sc) in enumerate(scores[:TOP_K], start=1):
            supporting_genes = [g for g in compound_to_genes.get(c, []) if d in gene_to_diseases.get(g, set())]
            rows.append({"disease_gid": d, "compound_gid": c, "score": sc, "rank": rank, "supporting_genes": ";".join(map(str, supporting_genes))})
    rank_df = pd.DataFrame(rows)
    out_rank = os.path.join(OUT_DIR, "ranked_candidates.csv")
    rank_df.to_csv(out_rank, index=False)
    print("Saved ranked candidates to", out_rank)

print("Pipeline finished. Check", OUT_DIR)
